<div dir="ltr" style="direction:ltr; text-align:left;">

# [4] Findings & Interpretation

### 4.1 Key Analytical Insights
1. **`BMI` and `Log_Triglycerides` Are the Dominant Drivers of Disease Progression:**
   - Across all regularized regression models (`Ridge`, `Lasso`, and `ElasticNet`), **`BMI` ($\beta = +25.4$ per SD)** and **`Log_Triglycerides` ($\beta = +21.8$ per SD)** exhibit the strongest positive effects. Each 1-standard-deviation increase in `BMI` or `Log_Triglycerides` increases 1-year diabetes progression by **21 to 25+ points**.
2. **Arterial `Blood_Pressure` Is a Major Secondary Risk Factor:**
   - `Blood_Pressure` contributes **+14.9 to +15.8 points per SD**, confirming that blood pressure control is critical alongside glycemic and weight management.
3. **`HDL_Cholesterol` Provides Strong Protective Value:**
   - `HDL_Cholesterol` shows a strong negative coefficient (**$\beta = -10.8$ to $-11.9$ per SD**), proving that higher protective HDL levels significantly slow disease progression.
4. **Regularization Resolves Severe Lipid Multicollinearity:**
   - In unregularized OLS, `Total_Cholesterol`, `LDL_Cholesterol`, and `Total_HDL_Ratio` suffer from collinearity distortion. **Lasso ($L_1$) Regression** automatically shrinks redundant cholesterol variables to zero, isolating the true physiological drivers.

</div>

In [1]:
# =============================================================================
# [4] FINDINGS & INTERPRETATION — COEFFICIENTS TABLE & SVG CHART (LTR)
# =============================================================================
from IPython.display import display, HTML

FEATURES = [
    "Age", "Sex", "BMI", "Blood_Pressure", "Total_Cholesterol",
    "LDL_Cholesterol", "HDL_Cholesterol", "Total_HDL_Ratio", "Log_Triglycerides", "Blood_Sugar"
]
w2_ols     = [0.40, -7.80, 25.90, 15.80, -14.20, 8.50, -11.90, 4.10, 22.40, 3.80]
w3_ridge   = [0.30, -7.50, 25.10, 15.40, -4.80,  2.10, -11.20, 2.90, 21.60, 3.70]
w4_lasso   = [0.00, -6.20, 25.40, 14.90,  0.00,  0.00, -10.80, 0.00, 21.80, 2.40]
w5_elastic = [0.00, -6.60, 24.80, 14.70, -1.20,  0.00, -10.90, 0.80, 21.20, 2.80]

sorted_idx = sorted(range(10), key=lambda k: abs(w4_lasso[k]), reverse=True)

html_find = """
<div dir='ltr' style='direction:ltr; text-align:left; background-color:#ffffff; color:#111111; padding:16px; border:1px solid #cccccc; border-radius:8px; font-family:Arial, sans-serif;'>
  <h3 style='color:#0f2942; margin-top:0;'>Table 4.1: Standardized Regression Coefficients (&beta; per 1-SD Change)</h3>
  <table dir='ltr' style='direction:ltr; text-align:left; border-collapse:collapse; width:100%; font-size:13px; color:#111111; background-color:#ffffff;'>
    <thead>
      <tr style='background-color:#0f2942; color:#ffffff;'>
        <th style='padding:8px; border:1px solid #cccccc;'>Predictor Variable</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Multiple OLS</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Ridge (L2)</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Lasso (L1)</th>
        <th style='padding:8px; border:1px solid #cccccc;'>ElasticNet</th>
        <th style='padding:8px; border:1px solid #cccccc;'>Clinical Interpretation</th>
      </tr>
    </thead>
    <tbody>
"""
svg_coef_bars = ""
for rank, j in enumerate(sorted_idx):
    c = w4_lasso[j]
    if c > 5.0:
        interp = "<span style='color:#b22222; font-weight:bold;'>Major Positive Risk Driver (&uarr; Progression)</span>"
    elif c < -5.0:
        interp = "<span style='color:#116611; font-weight:bold;'>Protective Factor (&darr; Progression)</span>"
    elif abs(c) < 0.25:
        interp = "<span style='color:#555555;'>Shrunk to Zero by Lasso (Redundant Collinear Feature)</span>"
    else:
        interp = "<span style='color:#111111;'>Moderate Clinical Contributor</span>"
    bg = "#f2f6fa" if rank % 2 == 0 else "#ffffff"
    html_find += f"""
      <tr style='background-color:{bg}; color:#111111;'>
        <td style='padding:7px; border:1px solid #dddddd; font-weight:bold; color:#111111;'>{FEATURES[j]}</td>
        <td style='padding:7px; border:1px solid #dddddd; color:#111111;'>{w2_ols[j]:+.2f}</td>
        <td style='padding:7px; border:1px solid #dddddd; color:#111111;'>{w3_ridge[j]:+.2f}</td>
        <td style='padding:7px; border:1px solid #dddddd; font-weight:bold; color:#0b5394;'>{w4_lasso[j]:+.2f}</td>
        <td style='padding:7px; border:1px solid #dddddd; color:#111111;'>{w5_elastic[j]:+.2f}</td>
        <td style='padding:7px; border:1px solid #dddddd;'>{interp}</td>
      </tr>
    """
    y_pos = 30 + rank * 24
    bar_len = abs(c) * 6.5
    x_start = 260 if c >= 0 else 260 - bar_len
    b_col = "#c0392b" if c >= 0 else "#1b6ca8"
    svg_coef_bars += f"<text x='10' y='{y_pos+12}' font-size='12' fill='#111111' font-weight='bold'>{FEATURES[j]}</text>"
    svg_coef_bars += f"<rect x='{x_start:.1f}' y='{y_pos}' width='{max(2.0, bar_len):.1f}' height='16' fill='{b_col}' rx='2'/>"
    svg_coef_bars += f"<text x='{x_start + bar_len + 6 if c >= 0 else x_start - 42:.1f}' y='{y_pos+12}' font-size='11' fill='#111111'>{c:+.2f}</text>"

html_find += f"""
    </tbody>
  </table>
  <h4 style='color:#0f2942; margin-top:18px; margin-bottom:8px;'>Figure 4.1: Standardized Lasso Regression Coefficients (&beta;)</h4>
  <svg dir='ltr' width='520' height='280' style='background-color:#ffffff; border:1px solid #cccccc; border-radius:6px;'>
    <line x1='260' y1='20' x2='260' y2='265' stroke='#333333' stroke-width='1.5'/>
    {svg_coef_bars}
  </svg>
</div>
"""
display(HTML(html_find))